# 案例 1 · 时间序列探索与分解 (Explore & Decompose)

对应讲义 [第 4 讲](https://jiangyou2025.github.io/kun/course/04/) 与 [第 5 讲](https://jiangyou2025.github.io/kun/course/05/)。

本案例用**合成数据**（无需下载任何数据集）演示：
1. 生成并可视化一条带趋势 + 季节性 + 噪声的时间序列；
2. 滚动均值 / 滚动标准差；
3. 手写加法分解（趋势 / 季节性 / 残差）；
4. 自相关函数 ACF。

> 依赖：`numpy`, `pandas`, `matplotlib`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(0)
n = 730                       # 两年的每日数据
t = np.arange(n)
trend  = 0.05 * t             # 线性趋势
weekly = 4 * np.sin(2*np.pi*t/7)        # 周季节性
yearly = 10 * np.sin(2*np.pi*t/365.25)  # 年季节性
noise  = np.random.normal(0, 2, n)
y = 20 + trend + weekly + yearly + noise
idx = pd.date_range('2022-01-01', periods=n, freq='D')
s = pd.Series(y, index=idx, name='value')
s.head()

## 1. 可视化原始序列

In [ ]:
plt.figure(figsize=(11, 4))
plt.plot(s.index, s.values)
plt.title('Synthetic daily time series')
plt.xlabel('date'); plt.ylabel('value')
plt.tight_layout(); plt.show()

## 2. 滚动均值与滚动标准差
趋势让滚动均值缓慢上升；滚动标准差大致平稳，说明方差比较稳定。

In [ ]:
roll_mean = s.rolling(30).mean()
roll_std  = s.rolling(30).std()
plt.figure(figsize=(11, 4))
plt.plot(s.index, s.values, alpha=0.4, label='series')
plt.plot(roll_mean.index, roll_mean, label='30d mean')
plt.plot(roll_std.index,  roll_std,  label='30d std')
plt.legend(); plt.title('Rolling mean & std')
plt.tight_layout(); plt.show()

## 3. 手写加法分解 (Additive decomposition)
`观测 = 趋势 + 季节性 + 残差`。这里用周期 7（周季节性）演示；
生产中可直接用 `statsmodels.tsa.seasonal.STL`。

In [ ]:
period = 7
trend_est = s.rolling(period, center=True).mean()       # 趋势：居中移动平均
detrended = s - trend_est
season_ix = np.arange(len(s)) % period
season_mean = pd.Series(detrended.values, index=season_ix).groupby(level=0).mean()
seasonal = pd.Series(season_mean.reindex(season_ix).values, index=s.index)
residual = s - trend_est - seasonal

fig, ax = plt.subplots(4, 1, figsize=(11, 8), sharex=True)
ax[0].plot(s.index, s);          ax[0].set_ylabel('observed')
ax[1].plot(s.index, trend_est);  ax[1].set_ylabel('trend')
ax[2].plot(s.index, seasonal);   ax[2].set_ylabel('seasonal(7)')
ax[3].plot(s.index, residual);   ax[3].set_ylabel('residual')
plt.tight_layout(); plt.show()

## 4. 自相关函数 ACF
在滞后 7、14、21… 处出现峰值，正是周季节性的指纹。

In [ ]:
def acf(x, nlags=40):
    x = np.asarray(x, float) - np.mean(x)
    var = np.dot(x, x)
    return np.array([np.dot(x[:len(x)-k], x[k:]) / var for k in range(nlags + 1)])

a = acf(s.values, 40)
plt.figure(figsize=(11, 4))
plt.stem(range(len(a)), a)
plt.axhline(0, color='k', lw=0.8)
plt.title('Autocorrelation (ACF)'); plt.xlabel('lag')
plt.tight_layout(); plt.show()

## 小结
- 序列 = 趋势 + 季节性 + 噪声，建模目标是抓住前两者、不去拟合噪声。
- 滚动统计和 ACF 是判断趋势、季节性、平稳性的第一手工具。
- 下一步见 [案例 2](02_forecasting_baselines.ipynb)：用基线模型做预测并评估。